# Phase 2: pretrain one language model

Trains a 24.6M-parameter decoder-only Transformer on one language's corpus.
Run it **once per language** by changing `LANGUAGE` below.

### Before running

* Accelerator: **GPU** (P100 or T4). Settings -> Accelerator.
* Add the tokenized data: **+ Add Data** -> search your datasets ->
  `lma-phase2-tokenized`.
* On the second and later sessions, also add your own checkpoint dataset so the
  run continues instead of starting over. See the resume section at the bottom.

### Why this is split across sessions

Kaggle stops a session after 12 hours. A full run needs more than that, so the
trainer checkpoints every 1,000 steps and stops cleanly before the limit. A fresh
session resumes from the last checkpoint and continues on exactly the same
trajectory - this is verified bit-exact by `scripts/test_model.py`.

## 1. Choose the language

`hi` is Model H (Hindi, higher-resource); `ne` is Model L (Nepali).

In [ ]:
LANGUAGE = "hi"        # "hi" or "ne"
TIME_BUDGET_HOURS = 10.5   # stop and checkpoint before Kaggle's 12-hour cutoff

assert LANGUAGE in ("hi", "ne")
print(f"training Model {'H (Hindi)' if LANGUAGE == 'hi' else 'L (Nepali)'}")

## 2. Unpack the project code

The `lma` package is embedded in this notebook rather than cloned: the GitHub
Classroom repository is private, so an unauthenticated clone from Kaggle fails.
The SHA-256 of each module is printed so it can be matched against the committed
source.

In [ ]:
import base64
import gzip
import hashlib
import json
import sys
from pathlib import Path

EMBEDDED_SOURCE = (
    "H4sIAB1Xl2oC/819jZPbxpHvv4K3riuTEghp144qR4ep+EtO6ixHz9rE9Uqr4mIJcIlbEKDxsbuMTv/79a97vgFyV3KSevZd"
    "vARmBjM9Pf3dPe9Pym36bLksqqJbLpPd/mQenZzEET/e1llehi8v+N/v8hW9bGZ1Ve6j8yat2nXdbPMmKra7Mt/mVZdn0bqp"
    "t9GuKbZFV9zm0ev9ed2sNlGZ7vOmTS6qi+r1Jm3z6Cyq11G3yalt/d/5qotorKsia6PLy6pKnNGfXF7G0Z/76+uiun6ZrvKI"
    "ZxityrRt8za+qNIqi9JqTwPls6u+KLso7TqaTFFX0VVZr27iqK2j/DZv9hE9b+smqnd5k3KDouJJ0Kfu0iaLdjToRVW00V1T"
    "YJCo7rsov9+VxaroaNWbvMmxCIEILYaXm6zqal1cAw5100WvMMFv+ZFu0PnQQqvJRRXRP9+mfZuWb/Jy/bWedSxvFLi/4SXI"
    "o5d5nr2UqaonDqB+fEXPppjUcpmW5XIZLaK30urixJ3TSayfjnzceet8zXnqTst57E2En7+7qHykEih5KPVK9pJf9LIlcXRX"
    "dBvaUYJ7SnixS5t0m3cEuFXdV11E3ybEUoiGvWs36S4X1Dqnn7SDxXUFbIzWxX3eRmnUpc113gHhmrq/3tA+nv3uVUR7UlTp"
    "VZnbT7QRIYZCMEIrwq32pgVy8IdkPjKLro6u8qjJsZV5xhiGJimhzYaGKlZRSbNsGWGiKr/nHtTiotrs6RvOF2nV1Ije0Yqv"
    "ckLyaJunbd/QEvmgpWusPVWzItRkHE+i801KmF629UW1TW/oUwXBqqbFY0X0sTZPcfAw+dSHML7VRZuia3nOV30G6ADqhO20"
    "5pK2lhrSTOhYdRv6Y4jyy+W672iShGUKn9OqqjsevzWtsrRL1UE1zdqsWHWxfRXTNuVlhj4X1Z/M44uK/+MeprnGNfz75wCM"
    "WGdd5YQeDpEq0+q6T68VzWAU4UNDq+7uannaCg0gwGpKRAhFm0bbAQAWHT2ihRKYuohWSMC8JfBc9WXa7NWM8pQA3WORgGdX"
    "3+RV8Q/sKTCM9zECsOu7ijai2fX8SSGDp0AdGeWSx1221PUyyor1GssCcWPc217lWSZ7koEq7Wg6QlRVW4yJvmaVdKSbgmad"
    "t2qa+Md+Yx791NOgDdNhM+Vdka/oMEUv9elS07RNEjtYteQZuCO1tH03tGS1DUKAW6/PJk8zt4ul1nhD337Vt3TMi9siy6PL"
    "aklLzy69EfBkLphB7eiot+h9GbQqqgpz+wv+Q9idEVFRLGdNhG2mSX6VEyo0NwnR23XalzgUdfTlvflybMfEP+jft31aRnyY"
    "YrNDDt1p8l/7opEjSYPl9yv64NhSGDpqN16l98W23+KodqAXZV5d05SBnAA9weVHOtFAJzrlBeCVlv7UDI4Q2gCB5RyDVNLZ"
    "YOJFR4Ixfcv8Ug1DJ/BuUxAGF+1wqRuAqAQ3Bxa3tLC8Ih6sJkdkpKgI9/jZpM1pvzw+cDl1Fps19Y4Qdx59vSN+StMCaMyU"
    "YwcP6CBepVdFSdPLW4Fw218xrvkzlHPgohf1WN7lxfWma+fRGz7IWEZR4bzYQ7RN6WzcC6fBe3WgFAWgOST+h96kIObOCY2e"
    "KDy8dLiHBiSBhQalY4OWLWh26g+3LtYWG+82dampsIsbRUpL+GWTM3soiZCkjT7xq7QhYQYtSKJpthoAlkaHR72oOnlojiw9"
    "Ifng9Ew/llMpT1/oh3LQ5OEXv/9SP1YnC8//J/oJVHfB/5H3LlJL19/pzxgUWJd1ijfPk1N54+3bVV2X9PK86fWQDAvvsRow"
    "XxMrIkzulMBKSFiup9Hsjzwhh/AVRJvoVSKLiv5D/8K6o/9DM5n7W0Skmzbw72nZ5983Td1M/Ne8ixcnarTJe2fsD9NoCwpG"
    "8gGImDDkq72CsW2LXx+m2C930OnYlBngwKNgUfjHa7KIviTMdCZzeLQ/LNx2n7h6GcosiX/q9SuyNw6irxj1w8ULEo+RZyxe"
    "kVUmHJpX9zuCM7U1egRO91UK0XQAV3nwJ2pIokO3txiEnVgSI7HIQ3jrQETO1WuSKngHiRa1O2giwlQus+UNAT6LnkUbIu7q"
    "DBpI5iQ9VB7uPXvmIp+LyoaULFnKtPOBzPS27ZoYM3s3mNr3LJkagXYWyMxxdNWAh0QZRBBCxVVNwlhF1NYIC6yK0NPe6HAi"
    "Vvvi6Z4+TxyeqLTIh9hDFqJYAHaInC9vrli0jVabnGWD9DrFMNwbgrgSQOl1uiv33pR+Zui1AXZ+i0WJrG5WEu3KHrL+5WVH"
    "Ymh5eRnd5PskBJTDjeLolkTPdeQdg1h+WNIZe4fGoc4gnuZMMSUmUTKPnruzZ8a9tFxnEd3S6XTOpGbBXhsZ0RBR6WH7fBa9"
    "1nIV0dG6bwjz7un/Le9qo8n/jaP/iqO/x9FfpyKgkLTrHit3OOaFKzBaBiGa/gge8RMxcRpKqDKPIsskYddl68S2K0V3Jhkm"
    "Gz0l6NB/HUq2LXfUhF8TxOX9mjD7aTSh3+N9Kv78IjrDwGfBS9r6JYOAGvAEnvI3nkqvQMBq7R4z86OxTH8Xsms6PeUSI6jP"
    "emA6HxMTQJbWTQ4mTserU1LNiMThAIxpCDEcgz0O7xMkug33nFGa+oQI9XQMg57qVT91V/Q0EmIT0KX3/sm6OAk+cUG6evAo"
    "DrsM54Bew6eDjjJPNJa/hiObbcKA+segmV0m2tlfg4blVkgutcJ/4+HiCc6yZPrDef3hCPeolkxu2+Pc45y3cMzmcJRlhBxh"
    "+tbM8p3LOQyQl6y92rmwpDWYzcsmFfxVMqgiwe0OlJQeO6qEywDojWPI00PiZPzagyt0e09ntIpy1N7l+S66I/qx2tQtnRXF"
    "BObKMMd6p0NUssweJWcYiNKORahLwdXSu3RvzUEK+dnGkdXOGdK8qvSPYwj9lTCXxQH424YOjBaq19uR8/OOjp55O3ZU3g12"
    "3hn5mTvycNu7egnRwBcUBrv9Jm+KtAQvgSbIbHhXE5Jqc4I2cDV99XmreDfNZVU32SHkFDOOfNecDbbYELQ2dWaniI2RSa7K"
    "Vuw+c54mzzewTA6m/nMOc5cSSVJn7oQFe0Jt4tikddcNltJXNxUEHGL8w0PFRibs1Ps1r3cNzZpmlCyXxua0lEbL5YfBeqnl"
    "5MmT98Ryb7n7DYkPGAF9k6LL6fhPQdBv8FCGIdk+MIA6VmDPCvq1b7Wa/PD6fNZ2+zKfemZ2bZMOTOzOifwrurMB/UfWF2E6"
    "55/fa4wyTwyLxxMgAT/9ThQ0kp6IiFxUfZtnSWRswzh+Ky0kOlbxmWvIZxMumw7A55qc5chYbJ/tKi1hHKqZh2Y9DIFoumI7"
    "NEki7Y08aOt1t03vrXVlyHl5gmlZeqb6GcmZGEy0fkEUJnEw7zFVKlplu02VnRTrIFIkdOiieoyHgGH9BuvCYblVwOk2bF1m"
    "/fby8huC4VXarTaA9zn96Iot20iUBUVTgineZ5eXF9XWt2fh+Yb6sVGMG1kt4xLMUJQR1kE+wT6rnpLwt3F+dnDW+L+SCqc9"
    "qqrB42TdVysxRKHFy0f5Q/CvWHZHnA8TGpRa94T5nrH3VV92haxXoQoIz8xYjAw7+h6WWA1asShlvOFFhx6CTrTnaVMWsHZq"
    "Exib2hPmZEBCYwojPiSHjJ1M1m7FOEr0kRSkJ6J/PvHxltQhDORY69SIRFZzDPiPvKkdQ9d+xA/Q5BBimHBn4gYABo4be8QQ"
    "4thAYkXH5y74xwwjbU+4NJkmprfD4lwbyUINaNTWoBGrtk4j3/rAjbSebZvpJ76C89cq1xY6IROABlQa3oa/0wZcs1UsiV4B"
    "aNS0IMJChKjIgA8rCFq1OyAdTZLR2xzcvMuHClPM1v00aulDJX9727OvZZ22nWfy5nX8enNLSzBUduKtOY6+IPE9eATNaaGe"
    "4e8QxpjLsSEfHi4YEErRUpnbZGBF2vXI6l04EcLoIntkRxfCf9uBItGWweNCopqi5RtihKANKQGYnRQtjiLJYTCnsvU16epJ"
    "lt+SAjp1x8NGE/zbFK4+IsFVHV03aVbAZhF9U9flnO2A0Ltg6nY5lDiSCROcbeNDvVCUi2bZBwYteUEHo9XLtAq4gb77KOv2"
    "u3wh3WCXnAa6RFak1yCMi1PnxRDY1wXwaynAmVycCHVbYroXJDsIEAmyLdpV3eJlSvrh1D30ii9N/KHj6H6uFnXODC1WgszS"
    "gMrYU3lM6c70oet3Zf7W7+z+UjbfoRUKdv39AwSaHVPNdWjRuYefBgweasltqg4liw7E+SbfxNE5wXx6eRnY5YeL+rokFFNC"
    "m8gAbTdTZNniydDsGPgc3E9v+Ovn+Hr0N5KIYAezas7ZcCyLjimhwJ42L5oQpaMp7GhoekVHaB/Dz0sCZAGDGmH/V1GZryGt"
    "rIcDZj0LuKw9soFTCTlEqJRor6xxcB+1pL1s823dPMaURisU0crxv9BC1blSUhetc2wvxEU+vvACgRxAE2rWV2Xeivk2JzTO"
    "jihfanDCyvuEvxhYvpTVlzgwjdXwTkHaZ3JxC2t1O4/MDIHM6scX8usLIvzmtR3411jk+YUh75P7adIStxcNx9Bd4lSLs4Dy"
    "vUEzcQGTtFmLvMYTApWzknB6X7DflyQkJZSSpKcCB9LOHVE0Yu4EfRciaJaTZg7mFk0w95updcVp5U0xrau8rO/m7nDRKEQ2"
    "Mox6IChODxygEDh+TW6L/E71cISB2OfnU4lvoaOWT07j6MwZBVT35jePgq25/eRRvN0yGshMaSBKjpv7cIj+pH9nNzj7PqDO"
    "XQPoZ6GV8PRZ+2vTTXiX6HPK38imHoIqMOOGjhThRk8C16xoN4S6YJzQClwb82fWuq0DTIrGVZ4iQjB6cGOVhpbPqh7OHQp2"
    "KxLGRVonNM1ukugXCflQ0izPNNZi7XVT3wmHFr2E9YzYF6qMvEt8uoOxX3trEe2g2TXNpqI10pAzS8CI5pUls2oP8Y2Pmn0r"
    "OhSndSfINiRQTnGtszHNlc9k9oto8itt4Y2DDLOzOJqdTqekQEHpSXiPfNTxUeVbq5TOo1lRrWH+Eq3K9Z3Txhj2wiEArRBi"
    "tRh3RMj8SfSG1GbRIs49bZXtlqIzcliEPeLbdA+Ntd0g2KgZW638kYjKsVwXZSlLc+SJt3NC3Pn5u1hMgiRsYE0XJ1Nv2cqa"
    "/zJRa5rIyEL6CHyDlgNpc4IfASh/Yet2DtfZFmtzSbU+UfbEnT9Emvba6v+n6DbYs7pCyEMFKV8IMXyBRmGYj39iQA6ZWCb+"
    "SSS+dt3XPew8Ztc0chHr5gCBlOTUambbRiBZYopk4uWJuCvWyIktEtvIm13jef+xxH1IyhJ3Fi45zDx47/W2eML8xCgak/10"
    "OjBwkTzCOwcjVihViVcC3HzqKvBOfOAhxf21OiezO/iTR+NuTFDWXR1EOUgcYPTD9z/+jUDe3eU5i3Tb2GjiRt9DnFynTTiu"
    "Z010enFT21AhBHgyfWFt21iGnRaTL+9JzDNiNgKDpnOxI6EzS0dbErWMAR26AYyn6S5dQZ3n8D+hhtKaDWCsi8in6pw9mDM2"
    "GCuzmhOeZ4E/U1ElbBvcIqRRxqHTC/OAxLkq8FiDxkzjV1b/uw0G69WjVFnelN+iGqsBPlLV/hjd1lWwRtUqUZicBwOl6FW6"
    "C6RmIvzBAzEcOmxP8JWPwVHv0PBov0yu87KfqH0gMXY69c6sG7176NDC/IJgavYbeoF8c4uWbKetvCNNsK/pxDLV6VMbbvla"
    "jzUx9qzoCaLFodQ8EdFZx3fFMLmJMyK9M0NNlXVMxQrh0LJeh/FEjLET5r6qX7SjbmIvF38swb6sWwmgIu2ByLWKaVITUxHI"
    "rbU2QG/BBGEy3GvK1FdEKQoWazI+4hm8Wxz7qA4nSzo6ppaIGMFn2+9mVo7PiD4012DvSfRaryXalemK+Ygxz0daWWWqwOvB"
    "ByuCOy8jX5PQL6NjdU2+xYdpKjBvcyAa0Sv6LBuJ/61koKyWp+rg6sX4tGA6YrOiDmN2Yek38oWzj/qCxCG4fMsM/P+lReXn"
    "vpIzpY8HzAfAxruGWKBYxlKL7bSYSocvBlSDJbRaqfmVCTdw5LeJ2TSiGsPFOaC8h3oePTVjDt9oYJsxz0CJBjTs3p+NS6e8"
    "MNJDhOpgQsqBuG8dQJtetXXZd7kTSRsGz8Lz5Znk26Lq27rIqKlWWpq6SxvlNSAi0YpgwedUomx1xL6OA8fBzSH7clRuai3O"
    "Mh7SFbIg/jdW0kMZxP7KLFvOvGklOkdIDlus2+4rTTRytgg5EzMJNCIeXLHb1w3ubYlEVtdq3So03Zo+L2Mj/lZO/C6pOwUM"
    "T2KNgEhsZkoazxB2P9evvyftjfaPQX5X9yVcirSGXV0SeGIVRAs/J2lAJGXxS1KFVhuCZSOBzErWU1YhFqmwJhDHfze1Uy4v"
    "7dkYGOSHsWCuW1aTLTfw7CglG40cGxtxxI49PiIP9NGWfxus1kofOac/EjJMPFlDEVpGkCXoFh3X69ySayZv0xEKv/4oCq8i"
    "fB4hgLqgZpnRt62rkFnV2A1P9k2enreqzZtbDqpohWyPhZ77Hu1wMCcCXQWdOzHn9zqY1sl7mRE1ExFDB5WHQzItUgE+M/l8"
    "oVJVmpzEppYIfKqDgVKJ9jKhNzSXuhmOCFGkyRHFgSSHZCQiWW1EosIHF6OnQL0deq/gSRDmIedNwX7cmIcWNobFiH7WvZeI"
    "X1gCBkm+bMVSazQyGc30bDui5luWAlkF1WGDJrIotwa7dLXqt32pCXowZFfv2JCmdFkWw74ylizJEbjaa4PhGScVyEGIbkii"
    "bD2PmPWMOOId1l4YyZXWcE2iC5vbaTNdsY8ZDMGoSmF1ZgTijCE2pNKzbGmRajINkJwOAtok8KZjJZOLE4BXb99J2J5F6irB"
    "5BIRrZcjEeRMu/FRcYUsnifPCehdhj/OPDPdmetRFQANh7NxSC3weRUGInmYNNkynZpbkjVG9YV7/MQrmNDkMDEhYWqYmG16"
    "wvZARMTKSQiCEJ4z2ulNTmALhTECZ9FKJPUqVzOJLcEKoRlCUjoo4I/BbjrYPdWFA3gLYZIjqQTux3hd5luBIp2XB5dgGNA/"
    "YxVOkGW/dRFUeLeJplxCAvQSRA7GYXLsuJezwwF2TJklgHeQJcTiZQ1NLdzINs9J6m/z7i3C8pnIeSGCOm73uX8G/dN3/Nz5"
    "axTdr1YLSHSi2RLa6shmsr2w6vPBqEU24RGmMgtaxWM7o3FCNHRihpiGrnSs+elCTZE2Li8nQ9lfohkfVrucU5OFGpgDaM7w"
    "bedjapVKTorDKRxS3GJHcxsNp3wtnI/TepX5rqyvi06Z++qdqBTKkaF1kRmL3UArWCAecojzYs957CILXeFDP7hZ/092TuP9"
    "kFFWRdfFbV65sXfD7R+ft+ThAHxgeFAUWqRp8kIZBg866L+tyxLSOmyfwlcPJP49wnv9HbaBWevlpXz+8tIa16zkNJWwRyyA"
    "GkwINdiYHQ892JeXZjbctCzExhtGAqhKAgRhPdgDXm3CLtpT7dR2juJ59EdXg3AE9k/KyQpTNN+ff1CZWK2TmRe9H//kh68O"
    "5GSVg9xTV2Nm5U/oJHNC+x0b/XYoH0tZ6010TiqKAYyjHBa0ANhMhJDna1VonkvkOWevWLv/1xy6B2se6OXKOIYYCTvJAHdH"
    "g+uOjr/K/maPI5HlKHOcqzYcVVRoBkoryfYSWhb4R7e+j9UBHtI2nZEK9gVstQegqETAJIHdEwFtXEkJwqYt/AR82oYenuSL"
    "k6y+JlGkyznL9+JErJ8XJ1uEFfJjanBxkvhGmzHhnCj8/VQbc4b65oQeTUdG8fTIyX3oXVRnC7Up3h33Nfk8U4R3zTNFFA+D"
    "iGLtlOTXk/ujVix1/AYE6kg4TwvFhGTgoYvTrB4KKy/akFEmS+a1aEUBWJiwLoJFg/cLVT8isX0WfUtiWjtTMUWum0x/Oe1s"
    "QGxnEFhYRDrQP7unpwTF27whyp9JfIFK0NBjqHho9lOL7Qv+N4T29eyVawdDqlWkJalV2Z4OGCmqEkVFCwr4hQLFy2SFdS3V"
    "ukZIneI27ASd0ZzVbxCdCdz88bCLmkiiYtLRbKQVpxPkS8Tc3C9mp8+fx0cIl8miOpHvI3dIywJ41qonLf+2iISn9pfNLlKh"
    "1TULdFpqgnR0nVcISM8/TTzapvfLKr9bSiUBTst2had8y9HufZPbzOzTxF04KbLLm5GMb6fJdUM8Zj8qSrHKQNpEkR0e4mFP"
    "2tc9vc7pM21L0guxGhJ1coRlQ2zY7rpHiVSvuWnUPVqyCkH35/pOyifIAxB/IQqhROYC9TsuZtG6x1JlLaswiyT6BnFb0Smz"
    "n10uQSQjgYBE6rmoBzO29AoRZqe0Z4xLbIqJ/sL4m0lG4qXsykBclO38OcdoCIFKtzsxSYjZ50Z822VxAzir+hP+EHq7z9Ob"
    "XJW+uUYMDHRCGOBQC0SNGvTUmPAG1hFxq0G5UnTGyC8IZJI4JwTLbDhv4yavHiEVnksyCvbZFHjQ5yfTao0wWtm6oyGJwtRu"
    "U0+JWWvnn5ZaWF2esIQHCvR8OgwRfkigCSyjPuJNxwj/Tu+X9hncEdWq7+aOl+Aq39fI3LUSGUeeVXU4nBbwbLamrmlk3AUi"
    "4NG3RLCKah060Q7YCLIc6KA7nx1FIoLEEuOJcPx2Hkezcbk0mr8L+YTDVid6HGRHakL8jocjzjB/d1EN2L5C3xH7B8FRYE6D"
    "K6YiqD2REKiYTXP4m+0MAW4T0ZsfYlZmQLZt3U8cEhFHp/ns9yNGLUgBOKwPWG041JJQ3UyZOt1MtkU14d4j7FEHdCUSiyXA"
    "CqF8YP5enJl69wd8fyy27CiA5XRskd5T1duCDpmNPNN8VE00ZhsQ05S8XZxORzZVKMsDoILBHoZbn9LkWySRZdH3f30jGyxP"
    "QBOLTp8E0fnGhmy7dA/FeNWp/IevlG7NklHX9Kx9tNqptS4aIq/0reRR8GEX4AEDqqZESV+1tJ6c9ndUrLFZDuu+LJcg7RP7"
    "qVjB7lBPp+Wo4fXQrKL/WTh9EzvDaLFQnxzsY3ZvFk5Am7ylB7EzyDvBiNPpI7ZfqhYQ7Z0YMNGOTKZj9uorkk5vRmQ7+rzN"
    "42Rr3Hh1xNfUXoW3zkU6ZiIKa9DWBJZgPiTA9Fu239tsVpuvqlPokFiqE+hY0f2mqNJm/x09bvPOqStYVLam4Dl+ihMzlh+I"
    "qOE/3nTsVeVs2qWalERPtdRqWTYHCgeOfNsr+qe/Fz7Nm/AJz8B5GEzFeWPnFNQRFPBzzq0L+m8UtCW9WcJ2ffMVvFV6d0yl"
    "QF2LreXCcRCiSAzsaTdOX3ACbJPuTXE0yIpgeCSAISNHyyWSCXJRqepoIlSQALQjgQ6TysVPTxw9o+0S9my8TwgAwEdila4x"
    "20pQBwrCCZUoOV+1gmQUIdmZxTpUi5oR/TpNzqIfvtGrEKdVq5KDEfuwSldcXapAfAE7t8CS5VOKYHG0JUclf2KBP8QjlcWV"
    "bvLazyYlgk06KZJHd2GGqRPpMUAwL8bjZwEdx0bMlE1Lw5HV01R2brhdXmRplndpUaogCInGF2ypG1DtipSKuW7/JHry5Fwr"
    "rY3y4Lah3vrkSfS1mokYCB1p5Wl0ioxepwpMG+0lAf3yEuTt7u18dvpOUmjo0Z4fEfuFnZKYjVG2Ly+Ri61drWZER4FHE/ke"
    "HMZKjB9Oxo1l0cO4rTCV2xpIzr7p9Xp2tZ/hWDDAWsKhiiPzcYZaRyy0eZZqkqbSIUMtsRD9RW0agtlZuY+yetVzvbyruq+y"
    "FAF5gOp36nEb3SHyxVrugP0FEEyZZSpwUAUIPz7Yk1zbnE4hWqacTFM3GSObhqC1AGtzHsfJQJyW5GGu+UpKhYT+wuOVIX6Z"
    "HTmWqkjGT55WrSNcmxXS6eyljCSEDUlAp55SDqlTQf9ICHps2AoO3hzDkjKNUxeHZdeQFpOb4m3PD3s3/7oDFDQYBTRYm6O9"
    "HVWrZSaEftSXsIiJFp8HQ0ndE+lxXGfG33pRTzCbCrbg/I1gMxwPg6gDLPcNG7ol6FGoyeeCjkoBROgdYYfKsmg42LYdCgMI"
    "jODEY3azI6PFScGi1V2X9RXn0RAxeyZkDk7n/CE9kuM3F7xjE/ztV3uD2GKaJRJRNfAJihvgJYH5p7p7iZMj3oC1qj2i9xCD"
    "rfF6roz9GPQDyeTj4TsC3YWzKV6NKDpJi4sTYsgSmBAZ/gSb3sy4JkvQzBb8iqCD8H4JUYH9trYnl/VNxZ/bz4WzcbvA9m7Z"
    "WDIWTcURR7uEOBoxzolZo9a86ZWgYOxMf5CCK4FTu0RYdKJi/Zf0HPUsszAeiPBz4sxgijJ6PnZ+ktfG7hAHY3yIiMsTHr8P"
    "PzePPyh+gghS1PS648M65rRZqwJLyuHjOHkO+mKE0NBH3UKKY150W71VnA+wIXVpw/qSDfw7ELo+gOFsgIez6NSdEx9XRfn4"
    "b5fOiSlFiOEC0uOup31+TNztMOD2O0Sdg+2ZNCxfdHtcRi3pK3t2OXIQGD2wc3bpNNyItIIXX1pnosvf9HB7aqZlD0zNlrfY"
    "KEMbyRSe3XJAerA1xgdBqJ0g5egaEQfPY7MdpGFjVgs72cDf9g1X4kEyPgheLLVPmo6td1yeaybaucCYSeY9K9Bg+kQKMj8b"
    "ri3rO1tYKndSrOjPnVu7T0lZfE45yD44RG8dbHrbRoQK2m3lH813SdqCNExoHAa8xPJwWq7A6N1osr4gg1ZKuaARg2AiExt4"
    "grg9LCoQ8rykLT9Jy7QjyS9o5hEdwXBUKVXYPTAGVnUltdE4Q5LzIZBDkmlxH5JymSq6vUG6QVfP1LAAdjgeD6FK/Wg7Hwk7"
    "twV2R1w/ge+NU7bpo0v54mRqCzvE3vTGzGaS7fYpvW0k+d4lGGLbQWTc8kr0sEE4/4COcJXjMQfFOIVRbosBCflLx4ZmZZfV"
    "1b5Z1IgZ3TLE+nD2CMrEq3jwsEYCMYwiE3WSXWLthiOk2d3Lggh2JdXUKSOyxXIN6m3MvJB8O350l5Y3rXvMkKyoMGMniCNC"
    "ly2tj0OhUtxc4zf00lE7/1FJ0YW4lu9Qw3MMn9RmsI9AKuDbwtVqRxNbYUBWZospc+p+MKLexF8kG6+WXBvHrucT9//H6tpB"
    "2k6ySdGoqghc4YT2eI6Iw/klT+84OW4QMzOo9DmsC+qHjhmrw8IvM8oqvu3zNHL4DaoB8+fgyT+FtDIQKQZ2NCn3HZrRzNf/"
    "uFC7MyY24zSGFj3Dft46cyzMzPhQFNbx4fCfwBz9EB/4l/GCESvnx/CEj+ELj+ANwXY9xB1+O3n+NBIdrAnnSdFpT300KLGI"
    "RtA2aKxxkBp7AuJ1ziL7kpWwh6ohspDSRD//9IMobfGwLOKgvqVSJpnSttbss26h8EiCYRcoml76kEdAlfbIvIWr1bT9VgJQ"
    "kYqjLAT0FP5u7fqsoUZhPM6kTl0OSL2RrNRXYupI1YUQbFlhLDWeDp4K/V9fcc550++0UOkFwqnoKJ2jeISUucmokCuvim5p"
    "AJAwdH22HG5TLFvgVIQcNVL8rICRCrNxd0/ViLraa/rr4cJlMqqHH5graPJgzvSgfQilelJBld1EFdKEq+yQAjQougvFDyEh"
    "hPNWj53Go7V5OXaEtm8SEvJh+6t9l3Nzq11iNXR02060kbCHkgA4aMWoBaNle4Vj6bGdLCOvaK5vuFfeikNuEwIZcXsNQ31N"
    "lJ/LJ0z6G7nmqYF5Q+sOLQ/TKYOlFC+XkIIsR2wfCs+CToL9z6P/Sq9R6e2mQBkSOk0tokogUrGscXpGQnLfqIC/NILLjDdV"
    "0m63ci5TSdVPoje1rWwCa5L18MAwXe3ZI4CMfOvvUVc4qNoqOpnAOEMUYULFzE2OOHlbqzCW0ThqHSKlQ8Z0cJfKRZvp2SPg"
    "KVIrvKh2BbJY+p0pt2LqExCsOYwTgl/NrgiXhIEgiYr/6eUm/7utw9qT2i9QbPNPuTXoEW6I40UttSstUZGYg5qVsZ+M+lgH"
    "3fh1Ro7HzM+5d9yE0ebA3UauuZmwMfGtxIFCgxrm6sYOncu+NNlKHAQrjVQbNn6jfnTD5FiHgb3IZ6oBcajl4Ua/k0aSiL4E"
    "hrbm9o/nSoZFRIv35vRs+Vy/lDOwzPJVuh+5H+Qq79JT9/l/2udn3vPf2TUvV2Wx80La1H1NpBMtmQPSH4Np8lulY5iZ6pf2"
    "CA8GOLWLIdo/eH2m3rlm+bPnZy+e/17fjkKcbeTuE2jhBTKV/OTyg+XIhRssUS0dwH6oKjnbUUE+WYiAOuYSoXzn1Fe2RP5o"
    "UQpfisOTUfz7mDLW//Si057jej5SJRqJPIeqQ0//WdWh8S9ma/zdE0BcmSNW6+u5Sy4G1dxlD35URzJiowP8gRjjcq5r6ciJ"
    "VNU65EMRHzIj4v4ixSNUArdO7v46S7efw5aPuKrZtmY3FnGAAjUHoRz2xObKgrlc4ci3EtoCu/SGmDMCEfnIf2XtwzRH5qel"
    "N3N2s2kB34l+LOGIVWGC/S7jKBqp7Q52hFyRf+RSfsNUAJE6HEn0rbNadcJMEV0udc+s3KnhIzmynJe/IWqkQMrXSxIskRuw"
    "rUkGL/dKyJG6PAoMCL5O4TEWYQTB+3pLUpXabxl34ObT9/Pg/R+w84lHSIf4Ri08Wow7Mrg3abZTFV4WDkOqpDk1BGcOpEWh"
    "NC2cBM3dYQzlHmknwwaDIvIMnfUjTp2cxiDBU03Wcr585XnyO0yeXuAOD4iohKQT/mMHG4EeQYeSORAYcCSutJ/zjSKTIYhm"
    "452ccxhEpExYInBSRA8cSlU8Ft0SHJtfvAPKT8SgqnKhGSFNDSv2nYkIVnRcvrG6tqao79CWiS9nl8ZOYRi+QEGEPicVZ9M3"
    "nAcnNRisDMEyidSMS5VPU9RXT8bbFmyxX+cclajvXtsQ+b8xFUa7mut3IWxEYS7X90VqqaRLQ3flIDgpS3pFOtYakclcnV7f"
    "NqPr2RBNWBU7pimVFhM465KDAlFiPtUFrfi+oO+0tAzxtlGne1DuQfjACrE1VS1CBWecxPT/hg2Ppm4qKlY3KmcWxE2qiMhv"
    "3yE7klGt8loHOdVNviKNIldlBkYSrMezOuE7/Rdmaxbq0qkE119NYds7MzmmH5nga2Cus2Tkg48KkNWbFHbVslzd71oblYZ/"
    "3kNrxrUr0EHVVpP+6kiReMHEynn2IT4wgp7A2CBEunQ/hTyfEQdDgRg5245/SxUMh3H+279997VSwKp6hlB3WvqdukCYcYeH"
    "MKGOfUaiQrtMb9OiBFed+ARvQGQmbuoF4BNHZbMYkL2YReR2wQSRpWimYvznGVFknsWC/1d5NI6qLhzE59G376Fj8y2xrB1L"
    "ZUsxSimD1rbIZusSMA00FiPtWDu3kl7ldHpvrojfLyGawx9ihXodcCzxxtJ0U8BatIcM1HZvIfbhgLPcNdEO/nWKyhb7BZpM"
    "tfJSlksReFpXpXgeFCaCDcOvcqj0MlbSfHtFbMI9rcAVuEYUlzl3YqrEnOBea4nfSxFjtf2So9/YVGfKQrMG6nQDuA50urUO"
    "pkE35nLHNFKnbd3TSSka7VdxTRzaHKGsYUyu1WUd7rWgyi9zeXlxgkNwcXJ5CaLHv2HONq7tI0FRB7OiNHCDa7LH4DpU4uMx"
    "SB5tFsoH8QignKCtIRTk5YH0qLAmEZcAFOv/QrsBCHITDUjJHjhAXSTRUodLhIXSGBEXivN19cT5VtjWwhDEzPwIWmkAopKz"
    "+jOsoLTm8knr6+C5ApwOm1I/p+Otku0N/S/4ByII2RURi2qzrG+GngnpahTexUAEtNBgshl+VVuNLW2cBEETr1jyJ91kVbRS"
    "aoAJy+kLJfMha5LvEEd1ROXrVfVhhJx6NZc5LCq/V1cesr8Mdqz17gvQcusfWO/wAamfagoxs+TXBFdjcK5Du8VFOrbkIVu4"
    "1mXfblRSmIoR9ov0bHdLqb+cyb7hiQiMLmKyW4rRcaQ/R2pZLxog88XZ8M5S51Ph7acGfCEjvVqfvlia16Hf7MAMzObgy2Zo"
    "PinOFE9fhGggkDUp7Ntd8gMBk+sThRFfGhqxKl6dLQbQNDC0E2Qf0OgEBhWraCOUzYgH2qRt2nXNhLvHHF3PL4dVe7yTr4Ap"
    "bZ1jMPXvJeT0nbTqU3BOAjO+7sXNfRbNZm4QwuxT/3k4R1aX1nMDxbQJz6Ophx05r3Alw8pLqWaOqeMq2KJjDe1giiVRl24P"
    "SzDUOc9x+IsJS3C4LccmmJAQv8agCqQwAcwVh9k52yv3bxEfZueChFGLYs6Spjwgfts+FIgqhD1Ma6y03VM7oGGBX9s/nZAA"
    "PhOabieuzTTMbM8l2d9XmuAINtn8miEkR8N2XEaRuBF1HI2wMHM3eZYOGXKOyzyMKUCGqBzbvqtXJL9PxvQajLLkuFJnWB1q"
    "6h/Wwyc7iCgYzRRsTaAe7xECTlTi+mIv+Y1IKn83TJvPTXkCLjAjgvGUzY+T6ZDrKdYOxuWiAELYl3oW/XYiI2tjEJyC6om1"
    "+xzzbGqBHXqUGToOw81eO+cI1UR22iS3HZxHvg5HEo652v42r8JIH/mqPZv8bYmo3nFOpJ1IdPZ8zG+qEFl5QfWCR64DZerm"
    "X6j426gbe5zT23xpB1XkDAYFG4KGinNtp+JcIRcNSNkvJGnn/pWJknLk+wM5sVepa2D7/o3QqkzyA25KFSvmeiV9V6QDurQN"
    "Nk9f6w2DDpdwcRyauFTaC3CQ8AsvmIJFoiDOgQisVBTlZGd0QlgF7qVQyVTKv8n1KxxqpRTnGdsLtetVXTqK5A194Z1GT3GQ"
    "4AaITlwk+qEb7MVXNMgNFqEv9wiV1nyYDj6zb1cQvThZ0ueul/QTcoTDnIGRfYXqwD4Dd4LI5NpMXIQZoj0PwYeFSQPvgThR"
    "hkfESdubB1L08Y4iKple8vOBLgCo6cBpFgz48fAIth0EzZ03w8Pu2hWCft67QU9lZgj6qKfD0ArHuhB0cV/Fo7uyXGlfldoc"
    "XZe0PgQ0UcRsN8M5D3WRu2tUeM0waUaXMhefx9ipTMPxDoYhccV0iS1SusZYTFIyviQ1Q4T4mHVZrTPxI8HGYmjoWOjOckYe"
    "6gJ53e/BOobXbcl5xY9QuIOqKR/ck8m5QZ7O+wxZHI7P+T1nhyS7ziUVn0VC61GKhOuO1A0HI0EDYHfcnLPdhdwj4gVx4UU2"
    "u+NeTCjdwawWybeaYEyVv565zITTfWp1d42E3XJMglUOtii2zlFHQB7Sxdbr4n5ycULTT+ilZ3UQqIHzTczFvtRm6g2XNDkH"
    "7YapU0r42KmwD81HMcqQj4aZc48JOgOSD28g1hfP2tCzgHMPg88s9ZUF48dEkpa26Y7IjHg7fBlT8V0uAGmqFP82PhH053ks"
    "HSKs5vnWMIV300NGkyN9HSbxzs90U21whiaWJ4wqpIpDHP6K7v1uKOOOmGfC0OR8t3CGYk7zLg6r5Rju4bT1uE3YxWMcTqeA"
    "2YTdFO9Y+OAxfAbeq5A8ubwj6OdzHOWCPWw48DuPENvpkcoaIRn2Iz0dqA2HPYoaDrk+9nlFPg591A7yLlnteoldlj5IEfz9"
    "9NgULPmfqvuDDxD40Tlxu3bAKuzk7PDv/GsoiipbCpddWvpl5H1l0RgQrR9ZMXDJlCrYpu5yJK7CNb1JkSpwH/J+QKNEtRhy"
    "Io8RSaOAEen8PhkAmQTSSmeyurX9HP3J+C9ms99sHeJoItZphdpz5ILv0gmSewTEw+LBA3eXfy2H53hB4OpD2S/eTDivZUX8"
    "lRNnnfyWHpd7wBirbodMpZ78iL/XbjDz+CR6k3c6guBOrFMEfvg8OIxHuL+kdrT1SHE85ajhqBiU+VPh7MhVakcup3bkCK53"
    "EZZ9FaB+q7zSwj0vL8cwSG4bFeeZYMoja3zJhXnunYJHE7C52CNPyx+Kk6Q1ug9P3dBhLlnVQxBKhcdA6ODG07H7ZvntxYkW"
    "pBlC77m5ygFGHDOU6veB4jOPwzRucZ1oEd9JXOSimYsBdR7LOTpmFOLYLbGxo0YB/sd9LVHjS8kWO95GWZaeJ89Huufs5B/J"
    "bQoAoGKkbDxpYAxr+BoOE1MXqI2BH0dbJNmJbkySVrxhr+dSXOwjm84vYJaDBIIKWU2Y+fJZ9IP2wPjxtzpPflusmnqmjayE"
    "DVu+1IDEea8gqIwlMXDGQpyTirXiG+pFx4I1R8r9EugQaYdzjKgUfSHwAGHtSuHkEaM6+BXpiRV9fywfCGD0dnIITfeKD9qq"
    "0QjQMfsn24VHEJYTwEPD79DOO3LQHmfg/ZcZeQ8Zen+Dsdcig9SV1Ljg3kVhnH5Faw2pcGUcGspDQqmZLPfNEj4mR6fP/3kW"
    "Hdzmi+oAqdS2H/yHjazTBLd1chX4kY20WPdUPhqYuEcKwuk5cSB49Mfo+Wg9uL9VEhOnSnKi7U5qd6n4ig0MpnWZKaMi307i"
    "uVEdB61cSWJdqbosDCC6GRbDDIHRy1yk/II9myPg0HkMSd8VZZtg1nx4l4iBXTp6oHfBQOzDZAA03yinwscPzsObOROmyYER"
    "Le2W1L0DLRylCg3FXuXFs0dPhKuNZYYa7DCoMtpKGI2bQhj6m53p/ocE2zoB/XBpjyFSjgT7kEni9k6HO45sI/O8yZGV0n/d"
    "iWtXjPocV478z4MCxoGigaLqDiWLP76IPzx77zPX+EM0Wl9EDcQwf+9uwDNvvvPky/UDIzTR+7KZJ2ck80TvAZBntKgv5snp"
    "+gPu7r151h7tz9jy/gAWYagX8+T5+sMrU2FuMAYiHSRW5FHVDY/IModlmoMtRoSmR2Kll8dyCC3p5DfFyvAZ4ySfHmyqjzIb"
    "RwbSU9ksSq6jYo0i46CfHqJ0rplcuyzVpz8ejyVf56CgPD+GONoQE71Xn3/7uX70+buH8ZbYQNDT+hzR/2z94Z+GcrjDR33J"
    "daq+I1H4kL/igNBxqDmivca+cGyUwFEp9i3Pruu1X7LFeam2/iMQfSTr6hC6H5iY8ZeOfNPRzAdVDyY+LVeK0JRkCVefP4rp"
    "rhkOXGd0xE9ZyCNh/JEH6uKi0tmfRxRQZbQ4cjreHwLds+jFcybIyBIRAOvUG75ql9p8dXBg6MswU5BytPJsC2w+sZAa6z89"
    "VLDC2a1xE/LH7uHj905sGEdoMzc4TpEfJsWxDBPqcofJMTcPWx5GrTGUElQy1hk4ostc8qEeT6w9Is2TGiHRTIedlyNUeLSs"
    "08HN59Bif7HG9jpqdH1NsjVuwBFNC0RUm/KdewkRTPh5q+6f7pvbvB1xD425AJXVHBH9eps4jzromMiMUUYx2Aq0TrJ+u2tH"
    "jv37QwftcW733+Z+/+1u+MMOdq4OMDnkZo8OVsn+BN/5p8UEsP937AYSLjO5OI0PFqChP08+/C+R6vCOx6sAAA=="
)

SRC = Path("/kaggle/working/src")
modules = json.loads(gzip.decompress(base64.b64decode(EMBEDDED_SOURCE)))

for rel, text in modules.items():
    path = SRC / rel
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(text, encoding="utf-8")
    digest = hashlib.sha256(text.encode()).hexdigest()[:12]
    print(f"  {rel:32s} {len(text):>6,} bytes  sha256:{digest}")

sys.path.insert(0, str(SRC))
print(f"\nunpacked {len(modules)} modules")

## 3. Locate the data and confirm the GPU

In [ ]:
import torch

INPUT = Path("/kaggle/input")
candidates = sorted(INPUT.glob("*/*_train.bin")) + sorted(INPUT.glob("*/*/*_train.bin"))
if not candidates:
    raise FileNotFoundError(
        "No *_train.bin under /kaggle/input. Use '+ Add Data' to attach the "
        "lma-phase2-tokenized dataset."
    )

DATA_DIR = candidates[0].parent
print("data:", DATA_DIR)
for path in sorted(DATA_DIR.glob("*.bin")):
    print(f"   {path.stat().st_size / 1e9:6.2f} GB  {path.name}")

if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    total = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"\nGPU: {name}  {total:.1f} GB  bf16={torch.cuda.is_bf16_supported()}")
else:
    print("\nWARNING: no GPU. Set Accelerator to GPU in Settings, or this will take days.")

## 4. Build the model

Identical architecture for both languages, so any difference measured in Phase 2
reflects data scale rather than capacity. Only the vocabulary differs, and here
it happens to be 8,000 for both.

In [ ]:
from lma.model import ModelConfig, TransformerLM
from lma.train import TokenBinaryDataset, TrainConfig, Trainer

model_config = ModelConfig(
    vocab_size=8000,
    n_layer=12,
    n_head=6,
    n_embd=384,
    n_inner=1536,
    block_size=512,
    dropout=0.1,
    tie_weights=True,
)

model = TransformerLM(model_config)
counts = model_config.parameter_count()

print(f"parameters: {model.num_parameters():,}")
for key in ("token_embedding", "position_embedding", "blocks", "final_norm"):
    print(f"   {key:20s} {counts[key]:>12,}")
print(f"   embedding share:     {100 * model_config.embedding_share():.1f}%")

## 5. Train

The trainer resumes automatically when a checkpoint is present in the output
directory, and stops cleanly once the time budget is reached so the session ends
on its own terms rather than being killed mid-step.

In [ ]:
OUT = Path("/kaggle/working") / f"{LANGUAGE}_model"
OUT.mkdir(parents=True, exist_ok=True)

# Seed a checkpoint from a previous session's dataset, if one is attached.
for previous in INPUT.glob("*/*_model/checkpoint_latest.pt"):
    if previous.parent.name == OUT.name and not (OUT / "checkpoint_latest.pt").exists():
        import shutil

        for item in previous.parent.glob("*"):
            shutil.copy2(item, OUT / item.name)
        print(f"seeded from {previous.parent}")
        break

train_config = TrainConfig(
    batch_size=24,
    gradient_accumulation=4,
    learning_rate=6e-4,
    min_learning_rate=6e-5,
    warmup_steps=500,
    max_steps=12_000,        # keep FIXED across sessions: it sets the decay horizon
    weight_decay=0.1,
    grad_clip=1.0,
    eval_interval=500,
    eval_batches=100,
    checkpoint_interval=1_000,
    log_interval=50,
    seed=20260812,
    amp=True,
)

train_data = TokenBinaryDataset(
    DATA_DIR / f"{LANGUAGE}_train.bin", model_config.block_size, seed=train_config.seed
)
val_data = TokenBinaryDataset(
    DATA_DIR / f"{LANGUAGE}_validation.bin", model_config.block_size, seed=train_config.seed + 1
)

print("train:", f"{train_data.stats()['tokens']:,} tokens")
print("val:  ", f"{val_data.stats()['tokens']:,} tokens")

tokens_per_step = train_config.tokens_per_step * model_config.block_size
print(f"\n{tokens_per_step:,} tokens/step x {train_config.max_steps:,} steps "
      f"= {tokens_per_step * train_config.max_steps / 1e9:.2f}B tokens "
      f"({tokens_per_step * train_config.max_steps / train_data.stats()['tokens']:.2f} epochs)")

trainer = Trainer(model, train_data, val_data, train_config, OUT)
state = trainer.train(max_seconds=TIME_BUDGET_HOURS * 3600)

## 6. Where the run got to

In [ ]:
import math

print(f"step {state.step:,} / {train_config.max_steps:,}")
print(f"tokens seen: {state.tokens_seen / 1e9:.2f}B")
print(f"best val loss: {state.best_val_loss:.4f}  (ppl {math.exp(state.best_val_loss):.2f})")
print(f"wall time: {state.wall_seconds / 3600:.2f} h")

if state.history:
    print("\n   step      val_loss      ppl")
    for row in state.history[-10:]:
        print(f"  {row['step']:>7,}   {row['val_loss']:>8.4f}   {row['val_perplexity']:>8.2f}")

if state.step < train_config.max_steps:
    print("\nNot finished. Save this output as a dataset, attach it to a new")
    print("session alongside lma-phase2-tokenized, and run again to continue.")
else:
    print("\nTraining complete.")

## 7. Loss curve

In [ ]:
import matplotlib.pyplot as plt

if state.history:
    steps = [row["step"] for row in state.history]
    losses = [row["val_loss"] for row in state.history]

    figure, axes = plt.subplots(1, 2, figsize=(11, 4))

    axes[0].plot(steps, losses, marker="o", markersize=3)
    axes[0].set_title(f"Validation loss - Model {'H' if LANGUAGE == 'hi' else 'L'}")
    axes[0].set_xlabel("step")
    axes[0].set_ylabel("cross-entropy")
    axes[0].grid(alpha=0.3)

    axes[1].plot(steps, [math.exp(min(v, 20)) for v in losses], marker="o", markersize=3, color="tab:orange")
    axes[1].set_title("Validation perplexity")
    axes[1].set_xlabel("step")
    axes[1].set_ylabel("perplexity")
    axes[1].set_yscale("log")
    axes[1].grid(alpha=0.3)

    figure.tight_layout()
    figure.savefig(OUT / "loss_curve.png", dpi=150)
    plt.show()
else:
    print("no evaluations recorded yet")

## 8. A sample of what it writes

Not a graded metric - just a quick check that the model produces text in the
right script rather than noise.

In [ ]:
prompt = torch.zeros((1, 1), dtype=torch.long, device=trainer.device)
generated = trainer.model.generate(prompt, max_new_tokens=60, temperature=0.8, top_k=50)
print("token ids:", generated[0, :20].tolist(), "...")
print("\n(decode with the Phase 1 tokenizer during evaluation)")

## 9. Save the checkpoint

**Do not skip this.** `/kaggle/working` is wiped when the session ends.

Save the notebook version, then from the **Output** tab create a dataset named
`lma-phase2-<language>-checkpoint`. Attach it to the next session together with
`lma-phase2-tokenized`, and the run continues from where it stopped.

In [ ]:
for path in sorted(OUT.iterdir()):
    print(f"{path.stat().st_size / 1e6:9.1f} MB  {path.name}")

print(f"\nSave as a Kaggle Dataset named lma-phase2-{LANGUAGE}-checkpoint")